# Brain Tumor Detection and Classification using DenseNet121

Academic workflow for four-class Brain MRI classification.

> **Educational disclaimer:** This notebook is for educational and research purposes only and is not intended to replace professional medical diagnosis.


## 1. Problem Statement

Classify a supplied brain MRI image into one of four dataset categories: **Glioma**, **Meningioma**, **No Tumor**, or **Pituitary Tumor**. The principal architecture is ImageNet-pretrained DenseNet121.


## 2. Import Libraries

The notebook reuses the same project modules as the training and Streamlit application so preprocessing and class ordering stay consistent.


In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import *
from src.data_loader import load_datasets, validate_dataset_structure
from src.eda import run_eda
from src.model import build_densenet121_model, prepare_for_fine_tuning
from src.predict import predict_image

print('TensorFlow:', tf.__version__)


## 3. Configuration

Important values live in `src/config.py` rather than being duplicated across files.


In [ ]:
print('Image size:', IMAGE_SIZE)
print('Batch size:', BATCH_SIZE)
print('Classes:', CLASS_NAMES)
print('Model path:', MODEL_PATH)


## 4. Dataset Loading

Expected structure: `dataset/Training/<class>/...` and `dataset/Testing/<class>/...`. The training folder is split reproducibly into training and validation subsets; the supplied testing folder is reserved for final evaluation.


In [ ]:
if TRAINING_DIR.exists() and TESTING_DIR.exists():
    validate_dataset_structure()
    train_ds, val_ds, test_ds = load_datasets()
    print('Datasets loaded successfully.')
else:
    print('Dataset not found. Add dataset/Training and dataset/Testing before running data-dependent cells.')


## 5. Exploratory Data Analysis

EDA counts actual files, records observed dimensions, checks readability, examines class balance, and saves plots. No dataset statistics are hard-coded.


In [ ]:
if TRAINING_DIR.exists():
    stats = run_eda()
    display(stats)
else:
    print('Skipping EDA because the dataset is unavailable.')


## 6. Image Preprocessing

Pipeline: validate → RGB conversion → resize to 224×224 → float array → DenseNet `preprocess_input`. Grayscale MRI images are converted to three RGB channels because DenseNet121 expects three-channel input.


In [ ]:
from src.preprocessing import load_rgb_image, image_to_model_batch
# Example after adding an image:
# image = load_rgb_image('sample_images/example.jpg')
# batch = image_to_model_batch('sample_images/example.jpg')
# print(image.mode, image.size, batch.shape)


## 7. Data Augmentation

Only training images receive small random rotation, zoom, translation, and contrast changes. Validation/test data receive only DenseNet preprocessing.


In [ ]:
from src.preprocessing import create_data_augmentation
augmentation = create_data_augmentation(RANDOM_SEED)
augmentation.summary()


## 8. DenseNet121 Architecture

DenseNet connects layers densely so later layers can reuse earlier feature maps and gradients can flow efficiently. We remove the original ImageNet classifier and add a compact four-class head.


In [ ]:
model, backbone = build_densenet121_model()
print('Backbone trainable initially:', backbone.trainable)
model.summary()


## 9. Transfer Learning

**Stage 1:** keep the DenseNet121 backbone frozen and train the custom head.

**Stage 2:** reload the best Stage-1 model, unfreeze only selected upper DenseNet layers, keep BatchNorm frozen, and fine-tune at a much lower learning rate.


In [ ]:
# Inspect how fine-tuning would be configured:
# fine_tune_model = prepare_for_fine_tuning(model, FINE_TUNE_LAYERS)
# trainable_backbone_layers = sum(layer.trainable for layer in backbone.layers)
# print('Trainable backbone layers:', trainable_backbone_layers)


## 10. Model Training

The canonical training command is run from the repository root. It performs EDA, feature extraction, selective fine-tuning, checkpoint comparison, and saves the best real model.


In [ ]:
# Run in a terminal after adding the dataset:
# python -m src.train

# Or call from Python (training can be computationally expensive):
# from src.train import train
# train()


## 11. Fine-Tuning

Fine-tuning is deliberately selective rather than unfreezing the entire network. The lower learning rate reduces the risk of destroying useful pretrained features.


In [ ]:
print('Default upper layers considered for fine-tuning:', FINE_TUNE_LAYERS)
print('Fine-tuning learning rate:', FINE_TUNE_LEARNING_RATE)


## 12. Evaluation

After training, `python -m src.evaluate` calculates test accuracy, macro precision/recall/F1, a classification report, confusion matrix, and one-vs-rest ROC/AUC where valid. All values come from actual predictions.


In [ ]:
if EVALUATION_METRICS_PATH.exists():
    metrics = json.loads(EVALUATION_METRICS_PATH.read_text())
    display(metrics)
else:
    print('No evaluation metrics yet. Train and evaluate the model first.')


## 13. Confusion Matrix

Rows correspond to true categories and columns to predicted categories. Off-diagonal values reveal which categories are being confused.


In [ ]:
cm_path = PLOTS_DIR / 'confusion_matrix.png'
if cm_path.exists():
    from PIL import Image
    display(Image.open(cm_path))
else:
    print('Confusion matrix will appear after `python -m src.evaluate`.')


## 14. Classification Report

Precision measures how often predictions for a class are correct; recall measures how many true samples of a class are recovered; F1 balances both.


In [ ]:
if CLASSIFICATION_REPORT_PATH.exists():
    report = json.loads(CLASSIFICATION_REPORT_PATH.read_text())
    display(report)
else:
    print('Classification report not available yet.')


## 15. Prediction

Inference uses exactly the same DenseNet preprocessing and class order used by training.


In [ ]:
# After training and adding a sample image:
# result = predict_image('sample_images/example.jpg')
# display(result)


## 16. Grad-CAM

Grad-CAM highlights regions that influenced the neural network's selected class. It is an explanation of model attention, **not** clinical tumor localization.


In [ ]:
# After training:
# from app.predictor import load_model
# from app.gradcam import create_heatmap_images
# model = load_model()
# original, heatmap, overlay = create_heatmap_images('sample_images/example.jpg', model)
# display(original, heatmap, overlay)


## 17. Conclusions

This repository connects data inspection, consistent preprocessing, DenseNet121 transfer learning, selective fine-tuning, measured evaluation, prediction, and Grad-CAM in one reproducible academic pipeline. Any claims about model performance must be based on the generated evaluation artifacts from the actual dataset and trained model.


### Viva recap

- **CNN:** learns spatial image features using convolution filters.
- **DenseNet121:** a densely connected CNN used as the feature backbone.
- **Transfer learning:** adapts pretrained visual features to the MRI classification task.
- **Validation set:** guides training decisions without using the final test set.
- **Overfitting:** strong training performance but poor unseen-data performance.
- **Grad-CAM:** gradient-based visualization of influential image regions, not a diagnosis.
